In [6]:
import os
print(os.getcwd())

c:\Users\andre\Downloads\coding\gov_tech\eavs_clc\usda_ers


In [9]:
import polars as pl

timeseries = pl.read_parquet(r"..\data\cleaned\timeseries.parquet")

In [11]:
usda_ers = pl.read_csv(r"data\*.csv")
usda_ers = usda_ers.filter(pl.col("Attribute").str.contains(r"\d{4}$"))

usda_ers = usda_ers.with_columns(
    (pl.col("Attribute").str.slice(0, length = pl.col("Attribute").str.len_chars() - 5).str.to_lowercase()).alias("attribute"),
    (pl.col("Attribute").str.tail(4).cast(pl.Int16).alias("year")),
    (pl.col("FIPS_Code")).cast(pl.String).alias("fips_code"),
)

usda_ers = usda_ers.rename({"Value": "value"})

usda_ers = usda_ers.drop(["FIPS_Code", "State", "Area_Name", "Attribute"])

timeseries_usda_ers_subset = timeseries.select("fips_code", "year", "jurisdiction_name").join(usda_ers, how = "left", on = ["fips_code", "year"])

timeseries_usda_ers_subset = timeseries_usda_ers_subset.pivot(
    values = "value",
    index = ["fips_code", "year", "jurisdiction_name"],
    columns = "attribute",
    aggregate_function = "first"
)

timeseries_usda_ers_subset = timeseries_usda_ers_subset.drop(["null"])

timeseries_usda_ers_subset.write_parquet("timeseries_usda_ers_subset.parquet")

timeseries_usda_ers_subset

C:\Users\andre\AppData\Local\Temp\ipykernel_25336\3185605660.py:16: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  timeseries_usda_ers_subset = timeseries_usda_ers_subset.pivot(


fips_code,year,jurisdiction_name,civilian_labor_force,employed,unemployed,unemployment_rate,estimates_base,pop_estimate,n_pop_chg,births,deaths,natural_chg,international_mig,domestic_mig,net_mig,residual,gq_estimates_base,gq_estimates,r_birth,r_death,r_natural_chg,r_international_mig,r_domestic_mig,r_net_mig,median_household_income,med_hh_income_percent_of_state_total
str,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""02000""",2004,"""ALASKA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""",2006,"""ALASKA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""",2008,"""ALASKA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""",2010,"""ALASKA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""",2012,"""ALASKA""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""56045""",2014,"""WESTON COUNTY""",3925.0,3789.0,136.0,3.5,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""56045""",2016,"""WESTON COUNTY""",3960.0,3761.0,199.0,5.0,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""56045""",2018,"""WESTON COUNTY""",3784.0,3650.0,134.0,3.5,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [12]:
timeseries_with_usda_ers = timeseries.join(usda_ers, how = "left", on = ["fips_code", "year"])

timeseries_with_usda_ers = timeseries_with_usda_ers.pivot(
    values = "value",
    index = [i for i in timeseries.columns if i != "value"],
    columns = "attribute",
    aggregate_function = "first"
)

timeseries_with_usda_ers = timeseries_with_usda_ers.drop(["null"])

timeseries_with_usda_ers.write_parquet("timeseries_with_usda_ers.parquet")

timeseries_with_usda_ers

C:\Users\andre\AppData\Local\Temp\ipykernel_25336\3965995239.py:3: DeprecationWarning: The argument `columns` for `DataFrame.pivot` is deprecated. It has been renamed to `on`.
  timeseries_with_usda_ers = timeseries_with_usda_ers.pivot(


fips_code,state,state_abbr,jurisdiction_name,year,registered_eligible_voters,active_voters,inactive_voters,A1Comments,A2a,A2b,A2c,A2Comments,total_registrations_received,new_valid_registrations,pre_registrations,duplicate_registrations,rejected_registrations,intrajurisdiction_registration_updates,interjurisdiction_registration_updates,A3h_Other,A3h,A3i_Other,A3i,A3j_Other,A3j,A3Comments,total_forms_mail_fax_email,total_forms_in_person,total_forms_online,total_forms_dmv,total_forms_mandatory_nvra,total_forms_disability_agency,total_forms_armed_forces,total_forms_discretionary_nvra,total_forms_advocacy_groups,A4j_Other,…,F11d_1,F11d_2,F11d_3,F11d_4,F11d_5,F5_F11Comments,F12a,F12b,F12c,F12d,F12e,F12Comments,F13,civilian_labor_force,employed,unemployed,unemployment_rate,estimates_base,pop_estimate,n_pop_chg,births,deaths,natural_chg,international_mig,domestic_mig,net_mig,residual,gq_estimates_base,gq_estimates,r_birth,r_death,r_natural_chg,r_international_mig,r_domestic_mig,r_net_mig,median_household_income,med_hh_income_percent_of_state_total
str,str,str,str,i64,i64,i64,f64,str,i64,i64,i64,str,i64,i64,i64,i64,i64,i64,i64,str,i64,str,i64,str,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,…,i64,i64,i64,i64,i64,str,str,str,str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""02000""","""ALASKA""","""AK""","""ALASKA""",2004,472160,472160,null,"""ITEM NOT COVERED IN EAVS YEAR""",-66,-66,-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,-66,-66,-66,-66,-66,-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,-66,-66,-66,-66,-66,-66,-66,-66,"""ITEM NOT COVERED IN EAVS YEAR""",…,-66,-66,-66,-66,-66,"""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""","""ALASKA""","""AK""","""ALASKA""",2006,466258,466258,-99.0,null,-66,-99,-66,"""ITEM NOT COVERED IN EAVS YEAR""",235249,50487,-66,19143,6449,159170,-66,null,-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,"""ITEM NOT COVERED IN EAVS YEAR""",-66,null,26765,90650,-66,67736,119,53,110,1987,-66,null,…,-66,-66,-66,-66,-66,"""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""","""ALASKA""","""AK""","""ALASKA""",2008,495731,495731,74935.0,null,21433,-66,-66,null,271546,64971,-99,43293,4748,158534,-99,null,null,null,null,null,null,null,82737,70783,-99,76224,702,67,1147,34978,-99,"""Same Day Registration""",…,0,0,0,0,0,"""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""02000""","""ALASKA""","""AK""","""ALASKA""",2010,560146,494876,65270.0,null,-99,-66,-66,null,234426,48331,-99,51747,4320,130028,0,null,null,null,null,null,null,null,44282,101628,-99,75458,392,316,860,1680,9810,null,…,0,0,0,0,0,"""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS YEAR""","""ITEM NOT COVERED IN EAVS